<a href="https://colab.research.google.com/github/DataScyther/Customer-Segmentation-Retention-Analysis/blob/main/Customer_Segementation_%26_Retention_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [59]:
# Importing Python Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import matplotlib
import seaborn as sns
from matplotlib import colors
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from yellowbrick.cluster import KElbowVisualizer
from sklearn.cluster import KMeans
from mpl_toolkits.mplot3d import Axes3D
from sklearn.cluster import AgglomerativeClustering
from matplotlib.colors import ListedColormap
from sklearn import metrics
import warnings
import sys
if not sys.warnoptions:
  warnings.simplefilter("ignore")
np.random.seed(42)

In [60]:
#Loading the dataset
data = pd.read_csv("/content/marketing_campaign.csv" , sep="\t")
print("Number of datapoints:", len(data))
data.head()

Number of datapoints: 2240


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [61]:
#Informative on features
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   i

In [62]:
#To remove the NA values
data = data.dropna()
print('The total number of data-points after removing the rows with missing values are:', len(data))


The total number of data-points after removing the rows with missing values are: 2216


In [63]:
data["Dt_Customer"] = pd.to_datetime(data["Dt_Customer"], dayfirst=True)
dates = []
for i in data["Dt_Customer"]:
  i = i.date()
  dates.append(i)
data["Dt_Customer"] = dates

# Dates of the newest and oldest recorded customer
print("The newest customer enrollment data in the records:", data["Dt_Customer"].max())
print("The oldest customer enrollment data in the records:", data["Dt_Customer"].min())

The newest customer enrollment data in the records: 2014-06-29
The oldest customer enrollment data in the records: 2012-07-30


In [64]:
#Created a features "Customers_For"
days = []
d1 = max(dates)
for i in dates:
  delta = d1 - i
  days.append(delta)
data['Customers_For'] = days
data['Customers_For'] = pd.to_numeric(data['Customers_For'], errors="coerce")

In [65]:
if 'data' not in globals():
    print('Error: "data" is not defined. Please run the previous cells (data loading and cleaning) first.')
else:
    print("Total categories in the feature Marital_Status:\n", data["Marital_Status"].value_counts(), "\n")
    print("Total categories in the feature Education:\n", data["Education"].value_counts())

Total categories in the feature Marital_Status:
 Marital_Status
Married     857
Together    573
Single      471
Divorced    232
Widow        76
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64 

Total categories in the feature Education:
 Education
Graduation    1116
PhD            481
Master         365
2n Cycle       200
Basic           54
Name: count, dtype: int64


In [66]:
#Feature Engineering

# Check if the transformation has already been applied
if 'Year_Birth' in data.columns:
    #Age of customer today
    data['Age'] = 2025 - data['Year_Birth']

    #Total Spending on various items
    data['Spent'] = data['MntWines'] + data['MntFruits'] + data['MntMeatProducts'] + data['MntFishProducts'] + data['MntSweetProducts'] + data['MntGoldProds']

    #Deriving living situation by marital status
    data['Living_With'] = data['Marital_Status'].replace({'Married':'Partner', 'Together':'Partner', 'Absurd' : 'Alone', 'YOLO' :'Alone', 'Divorced': 'Alone', 'Single':'Alone', 'Widow':'Alone', 'Alone':'Alone'})

    #Feature indicating total children living in the household
    data['children'] = data['Kidhome'] + data['Teenhome']

    #Feature pertaining parenthood
    data['Is_parent'] = np.where(data.children > 0, 1, 0)

    #Segmentation education levels in three groups
    data['Education'] = data['Education'].replace({'Basic':'Undergraduate', '2n Cycle':'Undergraduate', 'Graduation':'Graduate', 'Master':'Postgraduate', 'PhD':'Postgraduate'})

    #For clarity
    data = data.rename(columns={'MntWines':'Wines', 'MntFruits' : 'Fruits', 'MntMeatProducts':'Meat', 'MntFishProducts':'Fish', 'MntSweetProducts':'Sweets', 'MntGoldProds':'GoldProds'})

    #Dropping some of the redundant features
    to_drop = ['Marital_Status', 'Dt_Customer', 'Z_CostContact', 'Z_Revenue', 'Year_Birth', 'ID']
    data.drop(to_drop, axis=1, inplace=True, errors='ignore')
    print("Feature engineering successfully applied.")
else:
    print("Features have already been engineered and original columns dropped. Reload the data if you need to run this step again.")

Feature engineering successfully applied.


In [67]:
data.describe()

,Income,Kidhome,Teenhome,Recency,Wines,Fruits,Meat,Fish,Sweets,GoldProds,...,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,Customers_For,Age,Spent,children,Is_parent
count,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,...,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2.216000e+03,2216.000000,2216.000000,2216.000000,2216.000000
mean,52247.251354,0.441787,0.505415,49.012635,305.091606,26.356047,166.995939,37.637635,27.028881,43.965253,...,0.073105,0.064079,0.013538,0.009477,0.150271,3.054423e+16,56.179603,607.075361,0.947202,0.714350
std,25173.076661,0.536896,0.544181,28.948352,337.327920,39.793917,224.283273,54.752082,41.072046,51.815414,...,0.260367,0.244950,0.115588,0.096907,0.357417,1.749036e+16,11.985554,602.900476,0.749062,0.451825
min,1730.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,29.000000,5.000000,0.000000,0.000000
25%,35303.000000,0.000000,0.000000,24.000000,24.000000,2.000000,16.000000,3.000000,1.000000,9.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.555200e+16,48.000000,69.000000,0.000000,0.000000
50%,51381.500000,0.000000,0.000000,49.000000,174.500000,8.000000,68.000000,12.000000,8.000000,24.500000,...,0.000000,0.000000,0.000000,0.000000,0.000000,3.071520e+16,55.000000,396.500000,1.000000,1.000000
75%,68522.000000,1.000000,1.000000,74.000000,505.000000,33.000000,232.250000,50.000000,33.000000,56.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,4.570560e+16,66.000000,1048.000000,1.000000,1.000000
max,666666.000000,2.000000,2.000000,99.000000,1493.000000,199.000000,1725.000000,259.000000,262.000000,321.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,6.039360e+16,132.000000,2525.000000,3.000000,1.000000


In [69]:
#To plot some selected features

#Setting up colors preferences
sns.set(rc={'axes.facecolor':'FFF9ED', 'figure.facecolor':'FFF9ED'})
pallet = ['#682F2F', '#9EF726', '#D6B2B1', 'B9C0C9', '#9F8A78', 'F3AB60']
cmap = colors.ListedColormap(['#682F2F','#9EF726F','#D6B2B1'],'#B9C0C9', '#9F8A78', '#F3AB60'])

#Plotting following features
To_plot = ['Income', 'Receny', "Customer_For","Age", "Spent","Is_parent"]
print("Relative plot of some selected features: A data Subset")
plt.figure()
sns.pairplot(data[To_plot], hue='Is_parent', palette=(["#682F2F","#F3AB60"]))
plt.show()

SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' (ipython-input-3730881456.py, line 6)